In [1]:
#Esimerkissä käytettävä synnytysdata ja siihen liittyvät esimerkit ovat Tarja Heikkilän kirjasta Tilastollinen tutkimus. 
#Kirjaa voi lainata Savonian kirjastosta. Myös e-kirja on saatavilla. Kirjan esimerkit on tehty SPSS-ohjelmistolla. Suosittelen tutustumista kirjaan, 
#jos tilastotiede kiinnostaa laajemmin. Tilastotieteen osaamisesta on todella paljon hyötyä data-analytiikassa. 

#Muista asentaa Scipy ennen esimerkkien kokeilemista. 

import pandas as pd
import numpy as np
import scipy.stats as stats

In [2]:
#Luetaan data. Nyt Excel-tiedostossa on kaksi laskentataulukkoa ja data on taulukossa syntyneet, joten annetaan myös taulukon nimi. 

df = pd.read_excel('c:\\users\\installer\\desktop\\syntyneet.xlsx', sheet_name = 'Syntyneet')

df


,htunvv,sukp,slkm,syntpaik,kans,sivs,avol,airask,aikesk,aisyn,...,ltmp7,ltmp8,lapsij,rkesto,ika,pailka,pitlka,ikalka,rkeslka,aisyn2
0,57,1,1,1,1.0,4.0,1.0,3,0,2,...,NaN,NaN,2.0,273,35,4,3,5,4,1
1,62,2,1,1,1.0,2.0,1.0,0,0,0,...,NaN,NaN,2.0,292,30,5,3,4,5,0
2,63,2,1,1,1.0,2.0,1.0,0,0,0,...,NaN,NaN,2.0,281,29,4,3,3,5,0
3,65,2,1,1,1.0,1.0,2.0,2,0,2,...,NaN,NaN,2.0,275,27,4,3,3,4,1
4,70,1,1,1,1.0,2.0,2.0,0,0,0,...,NaN,NaN,2.0,288,22,6,3,2,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
237,59,1,1,1,1.0,1.0,2.0,4,1,3,...,NaN,NaN,2.0,271,33,4,2,4,4,1
238,62,2,1,1,1.0,1.0,2.0,3,1,2,...,NaN,NaN,2.0,287,30,6,3,4,5,1
239,60,2,1,1,1.0,1.0,2.0,1,0,1,...,NaN,NaN,2.0,271,32,3,2,4,4,1
240,63,2,1,1,1.0,1.0,2.0,1,0,1,...,NaN,NaN,2.0,285,29,4,3,3,5,1


In [3]:
#Lasten syntymäpaino (g) on kerrottu sarakkeessa paino ja äidin aikaisempien synnytysten määrä sarakkeessa aisyn. 
#Ryhmitellään data siten, että yhdessä ryhmässä ovat ne lapset, joissa äidin aikaisempien synnytysten määrä on 0 ja toisessa
#ne lapset, joissa äidin aikaisempien synnytysten määrä on 1 tai suurempi. 

#Tehdään uusi dataframe, johon otetaan kaikki ne arvot painosarakkeesta, joissa samalla rivillä sarakkeen aisyn arvo on 0. 
esikoiset = df[df['aisyn']==0]['paino'] 

#Tehdään uusi dataframe, johon otetaan kaikki ne arvot painosarakkeesta, joissa samalla rivillä sarakkeen aisyn arvo on suurempi kuin 0. 

muut_lapset = df[df['aisyn']>0]['paino']

In [4]:
#Lasketaan seuraavaksi molempien ryhmien havaintojen lukumäärä, keskiarvo, keskihajonta ja keskiarvon keskivirhe. Pyöristetään tulokset
#kahden desimaalin tarkkuteen.Tämänkin voisi tehdä lyhyemmällä koodilla, esimerkiksi tekemällä funktion, joka laskee halutut tunnusluvut.
#Tulostetaan sitten kaikki tunnusluvut siististi taulukkoon --> Tehdään kaksi dataframea, jotka yhdistetään lopuksi.
#Taulukon arvojen perusteella näyttää siltä, että painoissa on selkeä ero. Pelkästään keskiarvojen perusteella ei kuitenkaan voi tehdä johtopäätöksiä, 
#vaan pitää tutkia sopivan testin avulla, että onko ero tilastollisesti merkitsevä. 

esikoiset_yv = pd.DataFrame({
                'Ryhmä':['Ei aikaisempia synnytyksiä'],
                'N': len(esikoiset),
                'Keskiarvo': [round(esikoiset.mean(),2)],
                'Keskihajonta': [round(esikoiset.std(),2)],
                'Keskiarvon keskivirhe': [round(esikoiset.sem(),2)]
            })


muut_lapset_yv = pd.DataFrame({
                'Ryhmä':['Aikaisemmin synnyttänyt'],
                'N': [len(muut_lapset)],
                'Keskiarvo': [round(muut_lapset.mean(),2)],
                'Keskihajonta': [round(muut_lapset.std(),2)],
                'Keskiarvon keskivirhe': [round(muut_lapset.sem(),2)]
            })


yhteenveto = pd.concat([esikoiset_yv, muut_lapset_yv])

print(yhteenveto.to_string(index = False))

                     Ryhmä   N  Keskiarvo  Keskihajonta  Keskiarvon keskivirhe
Ei aikaisempia synnytyksiä  81    3295.93        555.30                  61.70
   Aikaisemmin synnyttänyt 161    3609.75        578.37                  45.58


In [5]:
#Testataan ensin 95 % merkitsevyystqsolla, että ovatko ryhmien varianssit yhtä suuret. Käytetään tähän Levenen testiä. Nollahypoteesi on, että 
#varianssit ovat yhtä suuret.

varianssitesti = stats.levene(esikoiset, muut_lapset)

print(f"Levene'n testi: W = {varianssitesti.statistic:.2f}, p = {varianssitesti.pvalue:.4f}")

Levene'n testi: W = 0.09, p = 0.7684


In [6]:
#Nyt testin antama p-arvo on 0,7684, eli reilusti yli 0,05. Tällöin nollahypoteesi jää voimaan, eli varianssit voidaan olettaa
#yhtä suuriksi. P-arvo kertoo, että tekisimme mikäli hylkäisimme nollahypoteesin, tekisimme virheen 76,84 % todennäköisyydellä. 

#Testataan seuraavaksi, että painavatko ensimmäiset keskimäärin vähemmän kuin myöhemmin syntyvät lapset. Nollahypoteesi on siis: Ensisynnyttäjien lapset
#painavat yhtä paljon kuin muiden synnyttäjien lapset. Testaus suoritetaan yksisuuntaisena, koska esitetyssä väitteessa on arvio poikkeaman suunnasta. 

#Käytetään tähän kahden riippumattoman otoksen t-testiä ja oletetaan, että varianssit ovat yhtä suuret. 
#Päätetään ennen testin tekemistä, että käytetään 5 % merkitsevyystasoa. Lasketaan testisuureen arvo ja p-arvo sekä tulostetaan ne.

t_stat, p_arvo = stats.ttest_ind(esikoiset, muut_lapset, equal_var = True)
print(f"t = {t_stat:.2f}, p = {p_arvo:.8f}")

t = -4.04, p = 0.00007311


In [7]:
#Testisuureen arvo on nyt -4,09 ja p-arvo on 0,0001. P-arvo ennataan kaksisuuntaiselle testille (painot voivat poiketa kumpaan
#suuntaan tahansa). Yksisuuntaisen testin p-arvo saadaan jakamalla saatu arvo kahdella, p_yks = 0,000 / 2 = 0,000 = 0,000. 
#Tämä tarkoittaa sitä, että todennäköisyys virheelliselle johtopäätökselle, kun nollahypoteesi hylätään on 0,00 % kahden desimaalin
#tarkkuudella ilmoitettuna. Huomaa, että tilastollinen testi ei koskaan todista mitään väitettä oikeaksi 100 % varmuudella.

#Nollahypoteesi voidaan siis hylätä ja johtopäätös on, että ensisynnyttäjien lapset painavat vähemmän kuin muiden synnyttäjien lapset. Ero on 
#tilastollisesti merkitsevä 99 % merkitsevyystasolla, p = 0,000

In [8]:
#Lasketaan lopuksi vielä syntymäpainojen 95 % luottamusvälit. 95 % luottamusväli tarkoittaa väliä, 
#johon arvo sijoittuu populaatiossa 95 % todennäköisyydellä. Pandasissa tai scipy.statsissa ei ole tähän valmista funktiota, 
#joten tehdään sellainen itse. Funktio saa argumentteinaan datan ja halutun luottamusvälin desimaalilukuna. 

def luottamusvali(data, value):
    n = len(data)
    mean = np.mean(data)
    sem = stats.sem(data)
    t_arvo = stats.t.ppf(1-(1-value)/2, df=n-1)
    margin = t_arvo * sem
    return mean - margin, mean + margin

esik_ala, esik_yla = luottamusvali(esikoiset,0.95)
muut_yla, muut_ala = luottamusvali(muut_lapset,0.95)

# Luodaan taulukko ja tulostetaan se.

luottamusvalitaulu = pd.DataFrame({
    "Ryhmä": ["Ei aikaisempia synnytyksiä", "Aikaisemmin synnyttänyt"],
    "95 % Luottamusväli(g)": [f"{esik_ala:.2f} – {esik_yla:.2f}",
                           f"{muut_ala:.2f} – {muut_yla:.2f}"]
})

print(luottamusvalitaulu.to_string(index=False))


                     Ryhmä 95 % Luottamusväli(g)
Ei aikaisempia synnytyksiä     3173.14 – 3418.71
   Aikaisemmin synnyttänyt     3699.77 – 3519.73


In [9]:
#Tutkitaan seuraavaksi, että onko opinnäyteytyökyselyssä eroa eri opiskelijaryhmien mielipiteissä koskien opinnäytetyön ohjausta. 
#Verrataan toisiinsa liiketalouden ja tekniikan opiskelijoita. Suoraan ohjaajaan liittyvät seuraavat väittämät: Ohjaajani panos tuki työtäni, 
#Saamani ohjaus oli asiantuntevaa, Saamani ohjaus oli motivoivaa, Työni ohjaaja vastasi nopeasti tiedusteluihini, Ohjaustilanteet eivät tuntuneet
#minusta pelottavilta, Ohjaajaani oli helppo lähestyä ja Luotin ohjaajani neuvoihin. 

#Lasketaan näistä keskiarvo ja sijoitetaan keskiarvo uuteen muuttujaan nimeltä Tyytyväisyys ohjaukseen. 


df2 = pd.read_excel('c:\\users\\installer\\desktop\\opinnäytetyökysely.xlsx') #Luetaan data

df2['Tyytyväisyys ohjaukseen'] = df2[['Ohjaajani panos tuki työtäni', 'Saamani ohjaus oli asiantuntevaa', 'Saamani ohjaus oli motivoivaa',
                                    'Työni ohjaaja vastasi nopeasti tiedusteluihini', 'Ohjaustilanteet eivät tuntuneet minusta pelottavilta', 
                                     'Ohjaajaani oli helppo lähestyä','Luotin ohjaajani neuvoihin']].mean(axis=1)



In [10]:
#Lasketaan ensin keskiarvo kokonaistyytyväisyydestä ohjaukseen sekä tekniikan että liiketalouden opiskelijoille. 

liiketalous = df2[df2['Opiskeluala'] == 'Liiketalous']['Tyytyväisyys ohjaukseen'].mean()
tekniikka = df2[df2['Opiskeluala'] == 'Tekniikka']['Tyytyväisyys ohjaukseen'].mean()
#Tulostetaan arvot taulukkoon.

taulukko = {
    'Opiskeluala': ['Liiketalous', 'Tekniikka'],
    'Tyytyväisyys ohjaukseen': [liiketalous, tekniikka]
}

print(pd.DataFrame(taulukko).to_string(index=False))

Opiskeluala  Tyytyväisyys ohjaukseen
Liiketalous                 3.727679
  Tekniikka                 4.206349


In [11]:
#Taulukon perusteella vaikuttaisi siltä, että teknikan opiskelijat ovat tyytyväisempiä saamaansa ohjaukseen kuin liiketalouden
#opiskelijat. Varmistetaan, että onko asia näin. Koska mielipiteen kysyminen asteikolla 1-5 on järjestysasteikollinen, käytetään
#Mann-Whitneyn U-testiä. 

#Käytetään 95 % merkitsevyystasoa. Nollahypoteesi on, että opiskelijoiden kokonaistyytyväisyydessä ohjaukseen ei ole tilastollisesti 
#merkitsevää eroa.

#Testataan ensin, että onko ryhmien välillä eroa. Näillä parametreilla testi ei ota kantaa siihen, mihin suuntaan ero on.

u_stat, p_arvo = stats.mannwhitneyu(df2[df2['Opiskeluala'] == 'Liiketalous']['Tyytyväisyys ohjaukseen'],
                                    df2[df2['Opiskeluala'] == 'Tekniikka']['Tyytyväisyys ohjaukseen'], alternative='two-sided')

print(f"Mann–Whitney U-testi: U = {u_stat:.2f}, p = {p_arvo:.8f}")

Mann–Whitney U-testi: U = 1160.00, p = 0.00211928


In [12]:
#Nyt kaksisuuntaisen testin p-arvo on 0,002, eli nollahypoteesi voidaan hylätä. 95 %:n merkitsevyystasolla opiskelijoiden 
#kokonaistyytyväisyydessä ohjaukseen on tilastollisesti merkittävä ero. Testi ei kuitenkaan ota kantaa eron suuntaan. Tutkitaan
#seuraavaksi, että onko liiketalouden opiskelijoiden kokonaistyytyväisyys pienempi kuin tekniikan opiskelijoilla: alternatives = 'less'.
#Nollahypoteesi on edelleen, että kokonaistyytyväisyydessä ei ole eroa. 

u_stat, p_arvo = stats.mannwhitneyu(df2[df2['Opiskeluala'] == 'Liiketalous']['Tyytyväisyys ohjaukseen'],
                                    df2[df2['Opiskeluala'] == 'Tekniikka']['Tyytyväisyys ohjaukseen'], alternative='less')

print(f"Mann–Whitney U-testi: U = {u_stat:.2f}, p = {p_arvo:.8f}")

Mann–Whitney U-testi: U = 1160.00, p = 0.00105964


In [13]:
#Nyt p-arvo on 0,001, joten nollahypoteesi hylätään 95 % merkitsevyystasolla. Voidaan siis todeta, että liiketalouden 
#opiskelijoiden kokonaistyytyväisyys on pienempi kuin tekniikan opiskelijoilla. Tekniikan opiskelijat ovat siis tyytyväisempiä
#saamaansan ohjaukseen kuin liiketalouden opiskelijat. Tulos on tilastollisesti merkitsevä 95 % merkitsevyystasolla.

#Samaan tulokseen olisi päästy jakamalla kaksisuuntaisen testin p-arvo kahdella ja päättelemällä oikea suunta. 